In [2]:
import os
import random
import time
import joblib
import numpy as np
import polars as pl
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
# from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ART

from art.attacks.evasion import ProjectedGradientDescent, FastGradientMethod
from art.estimators.classification import TensorFlowV2Classifier, SklearnClassifier, XGBoostClassifier, LightGBMClassifier, CatBoostARTClassifier, KerasClassifier


I0000 00:00:1779195726.582674 1233492 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779195726.635333 1233492 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779195727.869676 1233492 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packa

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
# Carga Dataset UNSW-NB15

path_train = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv"
path_test  = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv"
TARGET_COL = "attack_cat"

df_train = pl.read_csv(path_train)
df_test  = pl.read_csv(path_test)

y_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

y_test = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

x_train = df_train.drop(TARGET_COL)
x_test  = df_test.drop(TARGET_COL)

X_full_train = x_train.to_numpy().astype(np.float32)
X_test_np = x_test.to_numpy().astype(np.float32)
y_full_train = y_train.to_numpy()
y_test_np = y_test.to_numpy()

y_full_train_01 = ((y_full_train + 1) // 2).astype(np.int8)
y_test_np01 = ((y_test_np + 1) // 2).astype(np.int8)

# Preparamos una sola vez las distintas vistas del dataset.
# Mantenemos el espacio original para restricciones tabulares y transferencia
# entre modelos que no comparten el mismo preprocesado.
mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_full_train).astype(np.float32)
X_test_scaled_mlp = mlp_scaler.transform(X_test_np).astype(np.float32)

cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_full_train).astype(np.float32)
X_test_scaled_cnn = cnn_scaler.transform(X_test_np).astype(np.float32)

DATASET_VIEWS = {
    "raw": {"train": X_full_train, "test": X_test_np},
    "standard": {"train": X_train_scaled_mlp, "test": X_test_scaled_mlp},
    "minmax": {"train": X_train_scaled_cnn, "test": X_test_scaled_cnn},
}

print(f"Train: {X_full_train.shape} | Test: {X_test_np.shape}")
print("Tras convertir -1/1 a 0/1, la clase 0 corresponde a Ataque y la clase 1 a Normal.")
print("Vistas disponibles:", ", ".join(DATASET_VIEWS.keys()))

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
TRAIN_DEVICE = "/GPU:0" if HAS_GPU else "/CPU:0"
INFER_DEVICE = "/CPU:0"

SEED = 42
MODEL_DIR = os.path.join("model", "unsw-nb15")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


Train: (175341, 12) | Test: (82332, 12)
Tras convertir -1/1 a 0/1, la clase 0 corresponde a Ataque y la clase 1 a Normal.
Vistas disponibles: raw, standard, minmax


In [4]:
# Modelos ganadores de UNSW-NB15

RF_CONFIG = {"n_estimators": 50, "max_depth": 23}
XGB_CONFIG = {"n_estimators": 200, "max_depth": 11, "learning_rate": 0.1}
LGBM_CONFIG = {"n_estimators": 100, "num_leaves": 145, "max_depth": 12, "learning_rate": 0.1}
CATBOOST_CONFIG = {"iterations": 500, "depth": 10, "learning_rate": 0.1}

SVM_C = 0.000187
MLP_INPUT_DIM = X_full_train.shape[1]
CNN1D_CONFIG = {"nf": 64, "k": 5, "d": 48}

def build_mlp_model(input_dim):
    model = keras.Sequential([
        keras.layers.InputLayer(input_shape=(input_dim,)),
        keras.layers.Dense(96, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

DEFAULT_CNN_DROPOUT = 0.2

def build_cnn1d_model(n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_CNN_DROPOUT):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def clone_keras_model_to_cpu(builder_fn, trained_model, *builder_args):
    with tf.device(INFER_DEVICE):
        cpu_model = builder_fn(*builder_args)
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model

In [5]:
# Carga de modelos entrenados para UNSW-NB15

MODEL_PATHS = {
    "rf": os.path.join(MODEL_DIR, "rf_unsw.joblib"),
    "xgb": os.path.join(MODEL_DIR, "xgb_unsw.joblib"),
    "lgbm": os.path.join(MODEL_DIR, "lgbm_unsw.joblib"),
    "catboost": os.path.join(MODEL_DIR, "catboost_unsw.joblib"),
    "svm": os.path.join(MODEL_DIR, "svm_unsw.joblib"),
    "mlp": os.path.join(MODEL_DIR, "mlp_unsw.joblib"),
    "cnn": os.path.join(MODEL_DIR, "cnn_unsw.joblib"),
}

def load_model(model_key):
    model_path = MODEL_PATHS[model_key]
    print(f"Cargando {model_key} desde {model_path}...")
    return joblib.load(model_path)

rf_model = load_model("rf")
xgb_model = load_model("xgb")
lgbm_model = load_model("lgbm")
cat_model = load_model("catboost")

print("Modelos cargados")

Cargando rf desde model/unsw-nb15/rf_unsw.joblib...
Cargando xgb desde model/unsw-nb15/xgb_unsw.joblib...
Cargando lgbm desde model/unsw-nb15/lgbm_unsw.joblib...
Cargando catboost desde model/unsw-nb15/catboost_unsw.joblib...
Modelos cargados


In [6]:
# Carga de SVM y modelos neuronales

svm_model = load_model("svm")
mlp_model = load_model("mlp")

X_train_tabular_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_tabular_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)

with tf.device("/CPU:0"):
    cnn_model = load_model("cnn")


Cargando svm desde model/unsw-nb15/svm_unsw.joblib...
Cargando mlp desde model/unsw-nb15/mlp_unsw.joblib...


I0000 00:00:1779195729.438333 1233492 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43487 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:4a:00.0, compute capability: 8.9
I0000 00:00:1779195729.439820 1233492 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 43487 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0000:61:00.0, compute capability: 8.9
I0000 00:00:1779195729.440893 1233492 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 43487 MB memory:  -> device: 2, name: NVIDIA L40S, pci bus id: 0000:ca:00.0, compute capability: 8.9
I0000 00:00:1779195729.442245 1233492 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 43487 MB memory:  -> device: 3, name: NVIDIA L40S, pci bus id: 0000:e1:00.0, compute capability: 8.9


Cargando cnn desde model/unsw-nb15/cnn_unsw.joblib...


Una vez que los modelos ya están entrenados, ya podemos entrar en la fase de Evaluación Adversaria. Primero vamos a ver que nuestro modelo recién entrenado rinda bien con x_test limpio

In [7]:
# Evaluamos en test MLP

mlp_model_cpu = clone_keras_model_to_cpu(build_mlp_model, mlp_model, MLP_INPUT_DIM)
def mlp_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)
def mlp_predict_scores(X):
    with tf.device(INFER_DEVICE):
        return mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    
def cnn_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = cnn_model.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)

# F1-score de referencia en test para MLP
y_test_pred_mlp = mlp_predict_labels(X_test_scaled_mlp)
mlp_f1 = f1_score(y_test_np01, y_test_pred_mlp, pos_label=0)
print(f"\nMLP F1-score en test: {mlp_f1:.4f}")

/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
W0000 00:00:1779195730.712366 1236868 platform_util.cc:146] Allowed device set contains 4 devices, but platform only sees 1
I0000 00:00:1779195730.730325 1236868 service.cc:153] XLA service 0x7f74f86350b0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779195730.730363 1236868 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1779195730.736762 1236868 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779195730.811151 1236868 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



MLP F1-score en test: 0.8410


In [8]:
# Extraemos restricciones tabulares a partir del train original
feature_names = x_train.columns

tabular_constraints_df = pl.DataFrame({
    "feature": feature_names,
    "min": x_train.min().row(0),
    "max": x_train.max().row(0),
})

feature_mins = tabular_constraints_df["min"].to_numpy().astype(np.float32)
feature_maxs = tabular_constraints_df["max"].to_numpy().astype(np.float32)

tabular_constraints = {
    feature: {"min": float(min_val), "max": float(max_val)}
    for feature, min_val, max_val in zip(feature_names, feature_mins, feature_maxs)
}

# 1. Restricciones para modelos que trabajan en espacio original
clip_values_raw = (feature_mins, feature_maxs)

# 2. Restricciones para el MLP, que trabaja con StandardScaler
feature_mins_mlp = mlp_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_mlp = mlp_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)
clip_values_mlp = (feature_mins_mlp, feature_maxs_mlp)

# 3. Restricciones para la CNN, que trabaja con MinMaxScaler
feature_mins_cnn = cnn_scaler.transform(feature_mins.reshape(1, -1)).reshape(-1, 1).astype(np.float32)
feature_maxs_cnn = cnn_scaler.transform(feature_maxs.reshape(1, -1)).reshape(-1, 1).astype(np.float32)
clip_values_cnn = (feature_mins_cnn, feature_maxs_cnn)

print("Restricciones tabulares extraidas para UNSW-NB15:")
display(tabular_constraints_df)

print(clip_values_mlp)

Restricciones tabulares extraidas para UNSW-NB15:


feature,min,max
str,f64,f64
"""dur""",0.0,59.999989
"""sinpkt""",0.0,84371.496
"""dinpkt""",0.0,56716.824
"""spkts""",1.0,9616.0
"""dpkts""",0.0,10974.0
…,…,…
"""rate""",0.0,1.0000e6
"""smean""",28.0,1504.0
"""dmean""",0.0,1458.0


(array([-0.20977475, -0.1361428 , -0.08937003, -0.14098223, -0.17204736,
       -0.05044967, -0.10392289, -0.57681924, -0.5313342 , -0.48070285,
       -0.48434597, -0.50301373], dtype=float32), array([  9.049153 ,  11.513798 ,  57.369225 ,  70.09933  ,  99.358185 ,
        74.135994 , 101.91599  ,   5.469112 ,   6.6800365,   5.1635404,
        47.911243 ,  37.04389  ], dtype=float32))


In [9]:
# Hay que tener en cuenta que si tuviéramos columnas One Hot Encoding, habría que asegurarse de que las restricciones 
# se apliquen correctamente y que ART no me genere un TCP = 0.62, ya que no tiene sentido que un valor One Hot Encoding tenga un valor intermedio entre 0 y 1. 
# En ese caso habría que indicarle a ART que esas columnas solo pueden tomar valores 0 o 1, y no valores continuos.

# ==========================================
# FASE 1. ENVOLVER EL MODELO (Caja Blanca)
# ==========================================

print("Envolviendo el modelo MLP en ART con restricciones tabulares...")

clasificador_art_mlp = KerasClassifier(
    model=mlp_model_cpu, 
    clip_values=clip_values_mlp, 
    use_logits=False
)

print("Envolviendo el modelo CNN en ART con restricciones tabulares...")

cnn_model_cpu = clone_keras_model_to_cpu(
    build_cnn1d_model,
    cnn_model,
    X_train_tabular_cnn.shape[1],
    CNN1D_CONFIG["nf"],
    CNN1D_CONFIG["k"],
    CNN1D_CONFIG["d"]
)

clasificador_art_cnn = KerasClassifier(
    model=cnn_model_cpu,
    clip_values=clip_values_cnn,
    use_logits=False
)


Envolviendo el modelo MLP en ART con restricciones tabulares...
Envolviendo el modelo CNN en ART con restricciones tabulares...


In [10]:
# ========================================================
# FASE 4. LANZAR ATAQUE FGSM PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

EPS_VALUES = [0.01, 0.03, 0.05, 0.075, 0.1, 0.2, 0.5]

MODELOS_TABLA = ["RF", "XGB", "LGBM", "CatBoost", "SVM", "MLP", "CNN"]

modelos_arbol = {
    "RF": rf_model,
    "XGB": xgb_model,
    "LGBM": lgbm_model,
    "CatBoost": cat_model,
}

modelos_clasicos = {
    "RF": rf_model,
    "XGB": xgb_model,
    "LGBM": lgbm_model,
    "CatBoost": cat_model,
    "SVM": svm_model
}


In [11]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_fgsm_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando FGSM sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, 0.0).astype(np.float32)

    ataque_fgsm_mlp = FastGradientMethod(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        batch_size=128,
    )

    # Generamos adversarios en el espacio del MLP
    x_test_fgsm_mlp = ataque_fgsm_mlp.generate(x=x_test_mlp_attack)

    # Preparamos las distintas transformaciones para evaluar los modelos en sus respectivos espacios
    x_test_fgsm_mlp_std = x_test_fgsm_mlp.astype(np.float32)
    x_test_fgsm_mlp_raw = mlp_scaler.inverse_transform(x_test_fgsm_mlp_std).astype(np.float32)
    x_test_fgsm_mlp_minmax = cnn_scaler.transform(x_test_fgsm_mlp_raw).astype(np.float32)
    x_test_fgsm_mlp_cnn = x_test_fgsm_mlp_minmax.reshape(
        x_test_fgsm_mlp_minmax.shape[0],
        x_test_fgsm_mlp_minmax.shape[1],
        1
    )

# Inicializar la fila con el valor actual del límite
    fila_resultados = {"Limite perturbacion": eps_base}

# 4. Bucle UNIFICADO para evaluar todos los modelos de la tabla
    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            # Modelos de árboles usan datos RAW
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_mlp_raw)
        
        elif nombre_modelo == "SVM":
            # SVM usa datos estandarizados
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_mlp_std)
            
        elif nombre_modelo == "MLP":
            # MLP requiere su función auxiliar
            y_pred = mlp_predict_labels(x_test_fgsm_mlp_std)
            
        elif nombre_modelo == "CNN":
            # CNN requiere su función auxiliar y datos minmax redimensionados
            y_pred = cnn_predict_labels(x_test_fgsm_mlp_cnn)
            
        # Calcular y guardar el F1-score directamente
        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_fgsm_mlp.append(fila_resultados)

print("Evaluacion FGSM asociada al MLP completada.\n")

Generando FGSM sobre MLP para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [15:02:16] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
 

Evaluacion FGSM asociada al MLP completada.



In [12]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_fgsm_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando FGSM sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, 0.0).astype(np.float32)

    ataque_fgsm_cnn = FastGradientMethod(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        batch_size=128,
    )

    with tf.device(INFER_DEVICE):
        x_test_fgsm_cnn = ataque_fgsm_cnn.generate(x=x_test_cnn_attack)

    x_test_fgsm_cnn_cnn = x_test_fgsm_cnn.astype(np.float32)
    x_test_fgsm_cnn_minmax = x_test_fgsm_cnn_cnn.reshape(
        x_test_fgsm_cnn_cnn.shape[0],
        x_test_fgsm_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_fgsm_cnn_raw = cnn_scaler.inverse_transform(x_test_fgsm_cnn_minmax).astype(np.float32)
    x_test_fgsm_cnn_std = mlp_scaler.transform(x_test_fgsm_cnn_raw).astype(np.float32)

# Inicializar la fila con el valor actual del límite
    fila_resultados = {"Limite perturbacion": eps_base}

# 4. Bucle UNIFICADO para evaluar todos los modelos de la tabla
    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            # Modelos de árboles usan datos RAW
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_cnn_raw)
        
        elif nombre_modelo == "SVM":
            # SVM usa datos estandarizados
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_cnn_std)
            
        elif nombre_modelo == "MLP":
            # MLP requiere su función auxiliar
            y_pred = mlp_predict_labels(x_test_fgsm_cnn_std)
            
        elif nombre_modelo == "CNN":
            # CNN requiere su función auxiliar y datos minmax redimensionados
            y_pred = cnn_predict_labels(x_test_fgsm_cnn_cnn)
            
        # Calcular y guardar el F1-score directamente
        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_fgsm_cnn.append(fila_resultados)

print("Evaluacion FGSM asociada a la CNN completada.\n")

Generando FGSM sobre CNN para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have va

Evaluacion FGSM asociada a la CNN completada.



In [13]:
import pandas as pd

tabla_f1_fgsm_mlp = pd.DataFrame(resultados_fgsm_mlp).set_index("Limite perturbacion")
tabla_f1_fgsm_mlp = tabla_f1_fgsm_mlp[MODELOS_TABLA].round(4)

tabla_f1_fgsm_cnn = pd.DataFrame(resultados_fgsm_cnn).set_index("Limite perturbacion")
tabla_f1_fgsm_cnn = tabla_f1_fgsm_cnn[MODELOS_TABLA].round(4)

print("Tabla FGSM asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_fgsm_mlp)

print("Tabla FGSM asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_fgsm_cnn)

Tabla FGSM asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.7290,0.7208,0.6652,0.7136,0.6677,0.7111,0.7294
0.030,0.7071,0.7202,0.6420,0.7091,0.6219,0.6851,0.6848
0.050,0.7082,0.6841,0.6411,0.7191,0.5590,0.7071,0.6765
0.075,0.7094,0.6774,0.6250,0.7020,0.4503,0.7080,0.6710
0.100,0.7125,0.6840,0.6015,0.6889,0.4097,0.7090,0.6680
0.200,0.7062,0.6608,0.5387,0.7181,0.3791,0.7097,0.6759
0.500,0.7112,0.6626,0.5239,0.7383,0.3811,0.7085,0.6995


Tabla FGSM asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.7255,0.7287,0.6826,0.6985,0.7431,0.7321,0.7156
0.030,0.7013,0.6789,0.6688,0.7076,0.7201,0.4889,0.7034
0.050,0.7067,0.5639,0.6706,0.6913,0.6822,0.4581,0.6882
0.075,0.7178,0.6979,0.6710,0.6299,0.6418,0.4983,0.6846
0.100,0.7194,0.6727,0.6745,0.6496,0.6396,0.5328,0.6847
0.200,0.7159,0.6564,0.6409,0.7293,0.6441,0.5298,0.6841
0.500,0.6806,0.6410,0.6032,0.7404,0.6424,0.5378,0.6847


In [14]:
# ========================================================
# FASE 5. LANZAR ATAQUE PGD PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

PGD_MAX_ITER = 20

print("Configurando PGD para varios limites de perturbacion...")
print(f"Iteraciones PGD: {PGD_MAX_ITER}")


Configurando PGD para varios limites de perturbacion...
Iteraciones PGD: 20


In [15]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_pgd_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando PGD sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, 0.0).astype(np.float32)

    ataque_pgd_mlp = ProjectedGradientDescent(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        max_iter=PGD_MAX_ITER,
        batch_size=128,
    )

    x_test_pgd_mlp = ataque_pgd_mlp.generate(x=x_test_mlp_attack)

    x_test_pgd_mlp_std = x_test_pgd_mlp.astype(np.float32)
    x_test_pgd_mlp_raw = mlp_scaler.inverse_transform(x_test_pgd_mlp_std).astype(np.float32)
    x_test_pgd_mlp_minmax = cnn_scaler.transform(x_test_pgd_mlp_raw).astype(np.float32)
    x_test_pgd_mlp_cnn = x_test_pgd_mlp_minmax.reshape(
        x_test_pgd_mlp_minmax.shape[0],
        x_test_pgd_mlp_minmax.shape[1],
        1
    )

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_mlp_raw)
        elif nombre_modelo == "SVM":
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_mlp_std)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_mlp_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_mlp_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_pgd_mlp.append(fila_resultados)

print("Evaluacion PGD asociada al MLP completada.")


Generando PGD sobre MLP para varios limites de perturbacion...


PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.50it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.56it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.52it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.55it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/pyt

Evaluacion PGD asociada al MLP completada.


In [16]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_pgd_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando PGD sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, 0.0).astype(np.float32)

    ataque_pgd_cnn = ProjectedGradientDescent(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        max_iter=PGD_MAX_ITER,
        batch_size=128,
    )

    with tf.device(INFER_DEVICE):
        x_test_pgd_cnn = ataque_pgd_cnn.generate(x=x_test_cnn_attack)

    x_test_pgd_cnn_cnn = x_test_pgd_cnn.astype(np.float32)
    x_test_pgd_cnn_minmax = x_test_pgd_cnn_cnn.reshape(
        x_test_pgd_cnn_cnn.shape[0],
        x_test_pgd_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_pgd_cnn_raw = cnn_scaler.inverse_transform(x_test_pgd_cnn_minmax).astype(np.float32)
    x_test_pgd_cnn_std = mlp_scaler.transform(x_test_pgd_cnn_raw).astype(np.float32)

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_cnn_raw)
        elif nombre_modelo == "SVM":
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_cnn_std)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_cnn_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_cnn_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np01, y_pred, pos_label=0)

    resultados_pgd_cnn.append(fila_resultados)

print("Evaluacion PGD asociada a la CNN completada.")


Generando PGD sobre CNN para varios limites de perturbacion...


PGD - Random Initializations:   0%|          | 0/1 [00:00<?, ?it/s]

PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  4.00it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  4.03it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  4.03it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  4.01it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/pyt

Evaluacion PGD asociada a la CNN completada.


In [17]:
tabla_f1_pgd_mlp = pd.DataFrame(resultados_pgd_mlp).set_index("Limite perturbacion")
tabla_f1_pgd_mlp = tabla_f1_pgd_mlp[MODELOS_TABLA].round(4)

tabla_f1_pgd_cnn = pd.DataFrame(resultados_pgd_cnn).set_index("Limite perturbacion")
tabla_f1_pgd_cnn = tabla_f1_pgd_cnn[MODELOS_TABLA].round(4)

print("Tabla PGD asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_pgd_mlp)

print("Tabla PGD asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_pgd_cnn)


Tabla PGD asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.7210,0.6873,0.7124,0.6579,0.7046,0.7165,0.7772
0.030,0.7196,0.7019,0.6780,0.6922,0.7054,0.7110,0.7162
0.050,0.7113,0.7075,0.7251,0.7095,0.6965,0.7103,0.7157
0.075,0.7093,0.7073,0.6508,0.6874,0.7105,0.7102,0.7103
0.100,0.7084,0.7000,0.6101,0.7091,0.7145,0.7102,0.7100
0.200,0.7089,0.6983,0.7188,0.7037,0.7102,0.7102,0.7083
0.500,0.7098,0.7087,0.4985,0.7102,0.7102,0.7102,0.7137


Tabla PGD asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.7247,0.7199,0.7669,0.6825,0.7444,0.7384,0.7156
0.030,0.6778,0.7015,0.7192,0.6880,0.6099,0.7157,0.7150
0.050,0.7013,0.7071,0.7073,0.7060,0.7053,0.7068,0.7102
0.075,0.7133,0.7124,0.7191,0.7003,0.6629,0.7126,0.7102
0.100,0.7160,0.6833,0.7201,0.7110,0.7142,0.6717,0.7102
0.200,0.7215,0.7111,0.7302,0.7036,0.6758,0.6134,0.7107
0.500,0.6781,0.7310,0.7148,0.6915,0.6761,0.6159,0.6918
